# Setup and config

In [ ]:
%cd vietnamese_traffic_law_RAG

/content/vietnamese_traffic_law_RAG


In [1]:
!git clone https://github.com/huynoob1312/vietnamese_traffic_law_RAG.git
%cd vietnamese_traffic_law_RAG
!pip install -r requirements.txt

fatal: destination path 'vietnamese_traffic_law_RAG' already exists and is not an empty directory.
/content/vietnamese_traffic_law_RAG


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 485 bytes | 242.00 KiB/s, done.
From https://github.com/huynoob1312/vietnamese_traffic_law_RAG
   a88375c..91ecc02  main       -> origin/main
Updating a88375c..91ecc02
error: Your local changes to the following files would be overwritten by merge:
	config.yaml
Please commit your changes or stash them before you merge.
Aborting


In [3]:
import os

DRIVE_DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/RAG_LEGAL"
REPO_PATH = "/content/vietnamese_traffic_law_RAG"

if os.path.exists(DRIVE_DATA_PATH):
    print("Đã tìm thấy Google Drive. Bắt đầu chép dữ liệu...")

    # Copy toàn bộ thư mục data (chứa các file luật)
    !cp -r "{DRIVE_DATA_PATH}/data" "{REPO_PATH}/"
    print("Đã chép xong thư mục 'data/'")

    # Copy TẤT CẢ các file benchmark (.jsonl) vào thư mục benchmark
    !cp "{DRIVE_DATA_PATH}/"*.jsonl "{REPO_PATH}/benchmark/" 2>/dev/null || true
    print("Đã chép xong các file Benchmark (eval_dataset*.jsonl)")

else:
    print("LỖI: Không tìm thấy đường dẫn trên Drive. Vui lòng kiểm tra lại DRIVE_DATA_PATH hoặc mount lại Drive!")


Đã tìm thấy Google Drive. Bắt đầu chép dữ liệu...
Đã chép xong thư mục 'data/'
Đã chép xong các file Benchmark (eval_dataset*.jsonl)


In [4]:
from google.colab import userdata
import os

os.environ['QDRANT_URL'] = userdata.get('QDRANT_URL')
os.environ['QDRANT_API_KEY'] = userdata.get('QDRANT_API_KEY')
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

print('Đã nạp thành công thông tin Qdrant Cloud!')

Đã nạp thành công thông tin Qdrant Cloud!


# Up to qdrant

In [ ]:
# !python benchmark/ingest_data.py

# Set up ollama qwen2.5 14b

In [ ]:
import subprocess
import time

!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

print("Đang khởi động máy chủ Ollama...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(4) # Đợi 4 giây cho server ổn định

!ollama pull qwen2.5:14b

print("THÀNH CÔNG!")

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Fetched 3,917 B in 1s (2,975 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependen

# Benchmark Retrieval and embedding

## BM25

In [5]:
%%writefile config.yaml
qdrant:
  collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "bm25" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" (200 câu) hoặc "full" (1500 câu)


Overwriting config.yaml


In [6]:
!python benchmark/run_benchmark.py

modules.json: 100% 387/387 [00:00<00:00, 1.06MB/s]
README.md: 100% 498k/498k [00:00<00:00, 36.6MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 128kB/s]
config.json: 100% 655/655 [00:00<00:00, 1.84MB/s]

model.safetensors: downloading bytes:  33% 155M/471M [00:02<00:04, 64.4MB/s, 12.6MB/s  ]
model.safetensors: downloading bytes:  41% 191M/471M [00:03<00:04, 67.3MB/s, 15.4MB/s  ]
model.safetensors: downloading bytes:  46% 215M/471M [00:04<00:05, 44.2MB/s, 17.2MB/s  ]
model.safetensors: downloading bytes:  67% 315M/471M [00:06<00:04, 32.2MB/s, 22.0MB/s  ]
model.safetensors: downloading bytes: 100% 315M/315M [00:08<00:00, 37.8MB/s, 21.9MB/s  ]
model.safetensors: reconstructing file: 100% 471M/471M [00:08<00:00, 56.5MB/s, 33.4MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 4539.69it/s]
tokenizer_config.json: 100% 443/443 [00:00<00:00, 1.56MB/s]

sentencepiece.bpe.model: downloading bytes:   5% 249k/5.07M [00:00<00:04, 1.12MB/s]
sentencepiece.bpe.model: downloading bytes: 1

### Latency CPU

## embedding: "intfloat/multilingual-e5-small"

In [11]:
%%writefile config.yaml
qdrant:
  collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "vector" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" (200 câu) hoặc "full" (1500 câu)


Overwriting config.yaml


In [13]:
!python benchmark/run_benchmark.py

Loading weights: 100% 199/199 [00:00<00:00, 4450.11it/s]
BẮT ĐẦU CHẠY BENCHMARK TRÊN 1501 CÂU HỎI...
Evaluating: 100% 1501/1501 [08:39<00:00,  2.89it/s]

KẾT QUẢ BENCHMARK (CẤU HÌNH HIỆN TẠI TỪ CONFIG.YAML)
R@10 (%): 79.51
R@100 (%): 85.84
MRR@10: 0.5527
MAP@10: 0.5523


## BM25 + embedding: "intfloat/multilingual-e5-small"

In [14]:
%%writefile config.yaml
qdrant:
  collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "hybrid" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" (200 câu) hoặc "full" (1500 câu)


Overwriting config.yaml


In [15]:
!python benchmark/run_benchmark.py

Loading weights: 100% 199/199 [00:00<00:00, 12050.34it/s]
BẮT ĐẦU CHẠY BENCHMARK TRÊN 1501 CÂU HỎI...
Evaluating: 100% 1501/1501 [09:43<00:00,  2.57it/s]

KẾT QUẢ BENCHMARK (CẤU HÌNH HIỆN TẠI TỪ CONFIG.YAML)
R@10 (%): 86.98
R@100 (%): 96.87
MRR@10: 0.6106
MAP@10: 0.6101


### Latency CPU

In [ ]:
!python benchmark/eval_latency.py

### Latency GPU

In [ ]:
!python benchmark/eval_latency.py

## embedding: "BAAI/bge-m3"

In [16]:
%%writefile config.yaml
qdrant:
  # collection_name: "legal_docs"
  collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  # embedding: "intfloat/multilingual-e5-small"
  embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "vector" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" hoặc "full"


Overwriting config.yaml


In [17]:
!python benchmark/run_benchmark.py

modules.json: 100% 349/349 [00:00<00:00, 1.19MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 412kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 28.5MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 176kB/s]
config.json: 100% 687/687 [00:00<00:00, 2.62MB/s]

pytorch_model.bin: downloading bytes:   5% 118M/2.27G [00:00<00:10, 198MB/s, 8.98MB/s  ] 
pytorch_model.bin: downloading bytes:  10% 225M/2.27G [00:01<00:10, 203MB/s, 19.8MB/s  ]
pytorch_model.bin: downloading bytes:  26% 590M/2.27G [00:03<00:10, 155MB/s, 47.7MB/s  ]
pytorch_model.bin: downloading bytes:  27% 612M/2.27G [00:03<00:12, 138MB/s, 49.1MB/s  ]
pytorch_model.bin: downloading bytes:  40% 910M/2.27G [00:05<00:12, 110MB/s, 64.6MB/s  ]
pytorch_model.bin: downloading bytes:  67% 1.52G/2.27G [00:11<00:18, 41.2MB/s, 76.9MB/s  ]
pytorch_model.bin: reconstructing file:  56% 1.27G/2.27G [00:15<00:11, 84.2MB/s, 30.3MB/s  ]
pytorch_model.bin: downloading bytes: 100% 1.52G/1.52G [00:21<00:00, 71.9MB/s, 7

## BM25 + embedding: "BAAI/bge-m3"

In [ ]:
%%writefile config.yaml
qdrant:
  # collection_name: "legal_docs"
  collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  # embedding: "intfloat/multilingual-e5-small"
  embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "hybrid" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" hoặc "full"


Overwriting config.yaml


In [ ]:
!python benchmark/run_benchmark.py

modules.json: 100% 349/349 [00:00<00:00, 1.30MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 795kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 35.4MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 388kB/s]
config.json: 100% 687/687 [00:00<00:00, 4.45MB/s]

pytorch_model.bin: downloading bytes:   5% 115M/2.27G [00:00<00:12, 176MB/s, 8.10MB/s  ] 
pytorch_model.bin: downloading bytes:   9% 206M/2.27G [00:01<00:10, 200MB/s, 18.3MB/s  ]
pytorch_model.bin: downloading bytes:  13% 298M/2.27G [00:01<00:08, 222MB/s, 24.8MB/s  ]
pytorch_model.bin: downloading bytes:  18% 414M/2.27G [00:02<00:08, 219MB/s, 35.9MB/s  ]
pytorch_model.bin: downloading bytes:  23% 526M/2.27G [00:02<00:09, 189MB/s, 44.2MB/s  ]
pytorch_model.bin: downloading bytes:  27% 619M/2.27G [00:03<00:08, 192MB/s, 49.8MB/s  ]
pytorch_model.bin: downloading bytes:  40% 909M/2.27G [00:04<00:06, 195MB/s, 69.7MB/s  ]
pytorch_model.bin: downloading bytes:  50% 1.14G/2.27G [00:06<00:09, 119MB/s, 78.6MB/s

## embedding: "keepitreal/vietnamese-sbert"

In [ ]:
%%writefile config.yaml
qdrant:
  # collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  # embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "vector" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" hoặc "full"

In [ ]:
!python benchmark/run_benchmark.py

## BM25 + embedding: "keepitreal/vietnamese-sbert"

In [ ]:
%%writefile config.yaml
qdrant:
  # collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  # embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "hybrid" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" hoặc "full"

Overwriting config.yaml


In [ ]:
!python benchmark/run_benchmark.py

modules.json: 100% 229/229 [00:00<00:00, 701kB/s]
config_sentence_transformers.json: 100% 122/122 [00:00<00:00, 611kB/s]
README.md: 100% 3.90k/3.90k [00:00<00:00, 5.16MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 328kB/s]
config.json: 100% 752/752 [00:00<00:00, 3.84MB/s]

pytorch_model.bin: downloading bytes:  51% 276M/540M [00:01<00:01, 230MB/s, 24.5MB/s  ]
pytorch_model.bin: downloading bytes:  95% 511M/540M [00:02<00:00, 343MB/s, 42.1MB/s  ]
pytorch_model.bin: downloading bytes: 100% 514M/514M [00:02<00:00, 182MB/s, 45.5MB/s  ]
pytorch_model.bin: reconstructing file: 100% 540M/540M [00:02<00:00, 191MB/s, 48.5MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 14489.23it/s]

model.safetensors: downloading bytes:   0% 0.00/540M [00:00<?, ?B/s]

tokenizer_config.json: 100% 313/313 [00:00<00:00, 1.58MB/s]
model.safetensors: downloading bytes:  35% 188M/540M [00:01<00:01, 246MB/s, 13.8MB/s  ]

vocab.txt: 100% 895k/895k [00:00<00:00, 13.4MB/s]
model.safetensors: downloadin

## BM25 + embedding: "bkai-foundation-models/vietnamese-bi-encoder"

In [ ]:
%%writefile config.yaml
qdrant:
  # collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  # embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "hybrid" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" hoặc "full"

Overwriting config.yaml


In [ ]:
!python benchmark/run_benchmark.py

modules.json: 100% 229/229 [00:00<00:00, 1.10MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 829kB/s]
README.md: 100% 6.46k/6.46k [00:00<00:00, 16.9MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 332kB/s]
config.json: 100% 777/777 [00:00<00:00, 3.40MB/s]

model.safetensors: downloading bytes:  46% 247M/540M [00:01<00:00, 318MB/s, 19.3MB/s  ]
model.safetensors: downloading bytes:  89% 482M/540M [00:02<00:00, 336MB/s, 39.9MB/s  ]
model.safetensors: reconstructing file:  50% 268M/540M [00:07<00:07, 34.8MB/s, 12.8MB/s  ]
model.safetensors: downloading bytes: 100% 514M/514M [00:08<00:00, 61.8MB/s, 45.9MB/s  ]
model.safetensors: reconstructing file: 100% 540M/540M [00:08<00:00, 64.9MB/s, 40.1MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 17555.66it/s]
tokenizer_config.json: 100% 1.17k/1.17k [00:00<00:00, 3.13MB/s]
vocab.txt: 100% 895k/895k [00:00<00:00, 34.8MB/s]
bpe.codes: 100% 1.14M/1.14M [00:00<00:00, 60.1MB/s]
added_tokens.json: 100% 22.0/22.0 [00:0

## Without multi query

In [ ]:
%%writefile config.yaml
qdrant:
  collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "hybrid" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "ollama" # "gemini" or "ollama"
  model: "qwen2.5:14b" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "sample" # "sample" (200 câu) hoặc "full" (1500 câu)


Overwriting config.yaml


In [ ]:
!python benchmark/run_benchmark.py

Loading weights: 100% 199/199 [00:00<00:00, 4681.93it/s]
BẮT ĐẦU CHẠY BENCHMARK TRÊN 199 CÂU HỎI...
Evaluating: 100% 199/199 [01:21<00:00,  2.43it/s]

KẾT QUẢ BENCHMARK (CẤU HÌNH HIỆN TẠI TỪ CONFIG.YAML)
R@10 (%): 87.69
R@100 (%): 95.48
MRR@10: 0.6208
MAP@10: 0.6171


## With multi query Qwen2.5 14B

In [ ]:
%%writefile config.yaml
qdrant:
  collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "hybrid" # "hybrid", "bm25", "vector"
  top_k: 100
  use_reranker: false
  use_multi_query: true
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 150

llm_params:
  provider: "ollama" # "gemini" or "ollama"
  model: "qwen2.5:14b" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "sample" # "sample" (200 câu) hoặc "full" (1500 câu)


Overwriting config.yaml


In [ ]:
!python benchmark/run_benchmark.py

Loading weights: 100% 199/199 [00:00<00:00, 829.66it/s]

[HỆ THỐNG] Đã tìm thấy Checkpoint! Khôi phục thành công 173 câu đã chạy.
BẮT ĐẦU CHẠY BENCHMARK TRÊN 199 CÂU HỎI...
Evaluating: 100% 199/199 [04:52<00:00, 11.25s/it]

KẾT QUẢ BENCHMARK (CẤU HÌNH HIỆN TẠI TỪ CONFIG.YAML)
R@10 (%): 87.94
R@100 (%): 96.98
MRR@10: 0.5505
MAP@10: 0.5506


## Multi query Qwen2.5 14B + Rerank